# baseline_rag.ipynb — Baseline RAG Pipeline
**Project:** Self-Corrective RAG (CRAG) — ITMD 524, Spring 2026  
**Purpose:** Verify the end-to-end retrieve → generate pipeline before full evaluation.



## Setup before running

1. **`.env` file** — place your Groq API key in `.env`:
   ```
   GROQ_API_KEY=your_key_here
   ```
   Upload it to the Colab session (Files panel → Upload to `/content/`) **or** store it at
   `/content/drive/MyDrive/.env` so it persists across sessions.

2. **FAISS artifacts** — must already exist on Google Drive from `offline_embed.ipynb`:
   ```
   MyDrive/crag-vectors/corpus_index.faiss
   MyDrive/crag-vectors/id_map.json
   MyDrive/crag-vectors/config.json
   ```

3. **Run all cells top-to-bottom.**

In [ ]:
# ── Cell 1: Drive Mount + All Constants ───────────────────────────────────────
import os
from google.colab import drive

drive.mount('/content/drive')

SAVE_DIR    = "/content/drive/MyDrive/crag-vectors"
INDEX_PATH  = f"{SAVE_DIR}/corpus_index.faiss"
IDMAP_PATH  = f"{SAVE_DIR}/id_map.json"
CONFIG_PATH = f"{SAVE_DIR}/config.json"

# ── Single source of truth for all configuration ──────────────────────────────
ENCODER_MODEL = "all-MiniLM-L6-v2"       # must match offline_embed.ipynb
GROQ_MODEL    = "llama-3.1-8b-instant"   # Groq-hosted Llama 3.1 8B
TOP_K         = 5                         # chunks retrieved per query

print(f"Save directory  : {SAVE_DIR}")
print(f"Encoder model   : {ENCODER_MODEL}")
print(f"Generator model : {GROQ_MODEL}")
print(f"TOP_K           : {TOP_K}")

## Step 1 — Install Dependencies

In [ ]:
# ── Cell 2: Install Dependencies ──────────────────────────────────────────────
!pip install -q faiss-cpu sentence-transformers python-dotenv groq

## Step 2 — Load Groq API Key

Upload your `.env` file to the Colab session (Files panel → Upload to `/content/`) **before** running this cell.  
Alternatively, if stored on Drive, change the path to `/content/drive/MyDrive/.env`.

In [ ]:
 # ── Cell 3: Load API Key from .env ────────────────────────────────────────────
from dotenv import load_dotenv

load_dotenv("/content/drive/MyDrive/.env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
assert GROQ_API_KEY, (
    "GROQ_API_KEY not found.\n"
    "Upload your .env file to /content/ and re-run this cell."
)
print(f"Groq API key loaded (starts with: {GROQ_API_KEY[:8]}...)")

## Step 3 — Load FAISS Index and ID Map

In [ ]:
# ── Cell 4: Load FAISS Index + id_map from Drive ──────────────────────────────
import faiss
import json

index  = faiss.read_index(INDEX_PATH)
id_map = json.load(open(IDMAP_PATH, encoding="utf-8"))

assert index.ntotal == len(id_map), (
    f"Mismatch: index has {index.ntotal} vectors but id_map has {len(id_map)} entries."
)

print(f"FAISS index loaded: {index.ntotal} vectors, dim={index.d}")
print(f"id_map loaded     : {len(id_map)} entries")

# Print build config for traceability
config = json.load(open(CONFIG_PATH))
print(f"\nCorpus config:")
print(f"  encoder_model    : {config['encoder_model']}")
print(f"  chunk_size_tokens: {config.get('chunk_size_tokens', 'N/A')}")
print(f"  n_source_chunks  : {config['n_source_chunks']}")
print(f"  n_sub_chunks     : {config['n_sub_chunks']}")
print(f"  build_date       : {config['build_date']}")

## Step 4 — Retriever

Encodes the query with `all-MiniLM-L6-v2` and performs exact L2 search against the 1158-vector FAISS index.  
Returns the top-k most similar sub-chunks with their text and metadata.

In [ ]:
# ── Cell 5: Encoder + retrieve() ─────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer(ENCODER_MODEL)

def retrieve(query: str, top_k: int = TOP_K) -> list[dict]:
    """Return top-k chunks for a query via exact FAISS L2 search."""
    q_vec = encoder.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = index.search(q_vec, top_k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        entry = id_map[str(idx)]
        results.append({
            "text":     entry["text"],
            "chunk_id": entry["chunk_id"],
            "score":    float(dist),   # L2 distance (lower = more similar)
        })
    return results

# ── Smoke test ────────────────────────────────────────────────────────────────
sample = retrieve("Apple renewable energy percentage", top_k=3)
print(f"retrieve() — {len(sample)} results")
for i, r in enumerate(sample, 1):
    print(f"  [{i}] {r['chunk_id']} | score={r['score']:.4f} | '{r['text'][:80]}'")

## Step 5 — Generator

Uses Groq's `llama-3.1-8b-instant` to generate an answer from retrieved chunks.  

**System prompt is minimal by design** — no "I don't know" escape hatch. The baseline must attempt
to answer even with irrelevant chunks so that hallucinations surface in the Faithfulness score
during the full evaluation phase. This is the failure mode the CRAG Router is designed to prevent.

In [ ]:
# ── Cell 6: Groq Client + generate() ─────────────────────────────────────────
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

SYSTEM_PROMPT = "You are a helpful assistant. Answer the question based on the provided context."

def generate(query: str, context_chunks: list[dict]) -> str:
    """Generate an answer from retrieved chunks using Groq / Llama 3.1-8B."""
    context  = "\n\n".join(c["text"] for c in context_chunks)
    user_msg = f"Context:\n{context}\n\nQuestion: {query}"
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        temperature=0,     
        max_tokens=512,
    )
    return response.choices[0].message.content.strip()

# ── Smoke test ────────────────────────────────────────────────────────────────
answer = generate("What is Apple's renewable energy percentage?", sample)
print("generate() — ok")
print(f"\nAnswer: {answer}")

## Step 6 — Full Baseline Pipeline

Chains retrieve + generate into a single function. Two smoke tests verify the end-to-end pipeline.

In [ ]:
# ── Cell 7: baseline_rag() — Full Pipeline ────────────────────────────────────
def baseline_rag(query: str, top_k: int = TOP_K) -> dict:
    """Full baseline pipeline: retrieve → generate (no validation)."""
    chunks = retrieve(query, top_k=top_k)
    answer = generate(query, chunks)
    return {
        "answer":   answer,
        "contexts": [c["text"] for c in chunks],  # list[str] required by RAGAS
        "chunks":   chunks,                         # full metadata for debugging
    }

# ── End-to-end smoke tests ────────────────────────────────────────────────────
queries = [
    "How does Apple track its carbon emissions?",
    "What percentage of Apple's suppliers have committed to clean energy?",
]

for q in queries:
    result = baseline_rag(q)
    print(f"Q: {q}")
    print(f"A: {result['answer'][:300]}")
    print(f"   → {len(result['contexts'])} chunks retrieved")
    print()

print("baseline_rag() — pipeline verified")